# Browse saved workups

Use this notebook to find results by readable sample descriptions rather than folder names.
Each row represents one saved export. Metadata comes from that export's `manifest.json`,
not from today's configuration CSVs. Refreshing reads metadata only, skips Zarr chunk
directories, and does not change or delete any results.

The default view shows **approved exports only**. Saving creates a `draft`;
approval means you explicitly reviewed that particular export, not that a fit passed.
`REVIEW_STATUS = None` includes all statuses. Older exports without review metadata
are `development`, never automatically approved. The development archive is outside
the default results folder; set `RESULTS_ROOT` to its archived `results/` folder to browse it.

`LATEST_ONLY = True` selects the newest export **within the chosen review status**.
A newer draft does not hide an older approved export. Other filters, including
reflection mode, apply after that latest selection. Latest is not a quality judgment.

Discovery checks that a manifest and finalized consolidated Zarr metadata exist.
It does not repeat the workup notebook's full array verification.
Malformed or unfinished exports are listed separately and excluded from the main table.
Older `longC`/`longO` labels are displayed as `long` plus the energy region;
the original manifest is unchanged. Unrecorded energy regions remain unknown.
## Independent reader setup

This notebook is self-contained: it does not import or require the workup package,
PyHyperScattering, a beamline connection, or the environment used to save the data.
You can copy this notebook to another computer and point it at copied results.

Run the helper cell below once, then the refresh, filter, and inspection cells.
Metadata browsing needs only pandas and Python's standard library.
The optional dataset-opening cell additionally needs xarray, Zarr 2, and numcodecs.
Use an ordinary Python kernel with these packages; exact writer versions are not required.
The project provides `requirements-browse.txt` for this lightweight reader.

Use a separate kernel from the processing notebook. Separate kernels do not share
variables, even if they use the same environment. The supplied browser cells are
read-only: they do not change configurations, rerun processing, or write exports.



In [ ]:
"""Browse saved RSoXS exports using manifest metadata, not image arrays.

The catalog is rebuilt on request. It does not modify exports, current
configuration files, or a persistent index. Latest means newest save time
within each scan_set_id; it does not mean scientifically preferred.
"""

import json
import hashlib
import os
from pathlib import Path
import warnings

import pandas as pd

# Reader-local path discovery: no installed workup package is needed.
def _results_base_dir():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if candidate.name == "results":
            return candidate.parent
        if (candidate / "results").is_dir():
            return candidate
    return here


RESULTS_BASE_DIR = _results_base_dir()


CATALOG_COLUMNS = (
    "sample_description", "sample_id", "sample_name", "polymer",
    "diameter_nm", "wash_state", "np_vendor", "mw_g_mol", "mn_g_mol", "pdi",
    "scan_set_id", "scan_type", "energy_region", "scan_type_recorded",
    "beam_center_reflection_mode",
    "review_status", "reviewed_at_utc", "reviewer", "review_note",
    "saved_at_utc", "run_id", "is_latest", "scan_ids", "polarizations",
    "scan_count", "pair_count", "group_count", "has_angular_fit",
    "schema_version", "git_commit", "git_working_tree_clean",
    "export_directory", "manifest_path", "zarr_store", "metadata_notes",
)


def _text(value):
    if value is None or isinstance(value, (dict, list)):
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value).strip()


def _number_label(value):
    text = _text(value)
    try:
        number = float(text)
    except ValueError:
        return text
    return f"{number:g}" if pd.notna(number) else ""


def _beam_center_reflection_mode(configuration):
    """Read the saved setting, never infer it from current code defaults.

    An explicit JSON null means no reflection, just as in beam centering.
    Missing or malformed metadata is unknown, not evidence of no reflection.
    The setting alone does not establish that center fitting was performed.
    """
    processing = configuration.get("processing")
    if not isinstance(processing, dict):
        return "not_recorded"
    fit_settings = processing.get("beam_center_fit")
    if not isinstance(fit_settings, dict) or "reflection_mode" not in fit_settings:
        return "not_recorded"
    mode = fit_settings["reflection_mode"]
    if mode is None:
        return "none"
    if isinstance(mode, str) and mode.strip():
        return mode.strip().casefold()
    return "not_recorded"


def _export_review_metadata(manifest_path, manifest):
    """Read a separate review history, failing closed on invalid decisions.

    Old manifests without a status are development exports, never approved.
    The latest decision must identify the exact unmodified saved manifest.
    """
    baseline = manifest.get("review_status", "development")
    result = {
        "review_status": baseline if isinstance(baseline, str) and baseline in {"draft", "development"} else "unknown",
        "reviewed_at_utc": None, "reviewer": None, "review_note": "",
    }
    reviews = manifest_path.parent / "reviews"
    try:
        if reviews.is_symlink():
            raise ValueError("Review history is a symbolic link")
        if reviews.exists() and not reviews.is_dir():
            raise ValueError("Review history is not a directory")
        events = sorted(reviews.glob("*.json"))
        if not events:
            return result
        latest = events[-1]
        if latest.is_symlink():
            raise ValueError("Review event is a symbolic link")
        event = json.loads(latest.read_text(encoding="utf-8"))
        if not isinstance(event, dict) or event.get("schema_version") != 1:
            raise ValueError("Unrecognized review record")
        if event.get("event_id") != latest.stem:
            raise ValueError("Review event identity does not match its filename")
        if event.get("scan_set_id") != manifest["scan_set_id"]:
            raise ValueError("Review belongs to a different scan set")
        digest = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
        if event.get("manifest_sha256") != digest:
            raise ValueError("Saved manifest has changed since its latest review")
        status = event.get("status")
        if status not in {"draft", "approved"}:
            raise ValueError("Unrecognized review decision")
        reviewer, note = event.get("reviewer"), event.get("note")
        if not isinstance(reviewer, str) or not reviewer.strip() or not isinstance(note, str):
            raise ValueError("Review is missing its reviewer or note")
        if status == "approved" and not note.strip():
            raise ValueError("Approval is missing its review note")
        timestamp = event.get("reviewed_at_utc")
        if not isinstance(timestamp, str):
            raise ValueError("Review is missing its timestamp")
        from datetime import datetime
        reviewed = datetime.fromisoformat(timestamp)
        if reviewed.tzinfo is None:
            raise ValueError("Review timestamp needs a timezone")
        if not latest.stem.startswith(reviewed.strftime("%Y%m%dT%H%M%S%fZ") + "_"):
            raise ValueError("Review timestamp does not match its event identity")
        return {
            "review_status": status, "reviewed_at_utc": timestamp,
            "reviewer": reviewer.strip(), "review_note": note.strip(),
        }
    except (OSError, ValueError, TypeError, KeyError) as error:
        result.update(review_status="unknown", review_note=f"Review needs inspection: {error}")
        return result


def _manifest_paths(root, issues):
    """Find manifests while pruning chunk stores and incomplete directories."""
    def walk_error(error):
        issues.append({"manifest_path": str(error.filename), "reason": str(error)})

    for current, directories, files in os.walk(root, topdown=True, followlinks=False, onerror=walk_error):
        directories[:] = sorted(
            name for name in directories
            if not name.startswith(".")
            and not name.endswith((".zarr", ".incomplete"))
            and not (Path(current) / name).is_symlink()
        )
        if "manifest.json" in files:
            yield Path(current) / "manifest.json"


def _catalog_row(manifest_path):
    with manifest_path.open(encoding="utf-8") as file:
        manifest = json.load(file)
    if not isinstance(manifest, dict):
        raise ValueError("Manifest must contain a dictionary")
    configuration = manifest.get("configuration", {})
    if not isinstance(configuration, dict):
        raise ValueError("Manifest configuration must be a dictionary")
    sample = configuration.get("sample", {})
    scan_set = configuration.get("scan_set", {})
    if not isinstance(sample, dict) or not isinstance(scan_set, dict):
        raise ValueError("Sample and scan-set metadata must be dictionaries")
    scan_set_id = _text(manifest.get("scan_set_id") or scan_set.get("scan_set_id"))
    if not scan_set_id:
        raise ValueError("Missing scan_set_id")
    configured_id = _text(scan_set.get("scan_set_id"))
    if configured_id and configured_id != scan_set_id:
        raise ValueError("Manifest and configuration scan_set_id disagree")
    generated = manifest.get("generated_at_utc")
    if not isinstance(generated, str) or not generated.strip():
        raise ValueError("Missing save timestamp")
    saved_at = pd.to_datetime(generated, utc=True, errors="raise")
    if pd.isna(saved_at):
        raise ValueError("Invalid save timestamp")

    store_name = manifest.get("zarr_store", "workup.zarr")
    if not isinstance(store_name, str) or not store_name:
        raise ValueError("Invalid Zarr store path")
    relative_store = Path(store_name)
    if relative_store.is_absolute() or ".." in relative_store.parts:
        raise ValueError("Zarr store path must remain inside the export directory")
    store = (manifest_path.parent / relative_store).resolve()
    try:
        store.relative_to(manifest_path.parent.resolve())
    except ValueError as error:
        raise ValueError("Zarr store points outside the export directory") from error
    if not store.is_dir() or not (store / ".zmetadata").is_file() or not (store / ".zgroup").is_file():
        raise ValueError("Missing finalized Zarr store or consolidated metadata")
    groups = manifest.get("zarr_groups")
    if not isinstance(groups, dict) or not groups:
        raise ValueError("Missing exported-group dictionary")
    if any(not isinstance(name, str) or not isinstance(variables, list) for name, variables in groups.items()):
        raise ValueError("Malformed exported-group dictionary")

    recorded_type = _text(scan_set.get("scan_type"))
    scan_type = recorded_type.casefold()
    region = _text(scan_set.get("energy_region"))
    notes = []
    # Older manifests used longC/longO before these fields were separated.
    # Normalize the catalog view only; never rewrite the historical manifest.
    if scan_type.startswith("long") and len(recorded_type) > 4:
        if not region:
            region = recorded_type[4:].upper()
        scan_type = "long"
        notes.append(f"Legacy scan_type {recorded_type!r} displayed as long/{region}")
    if not region:
        notes.append("Energy region was not recorded")
    reflection_mode = _beam_center_reflection_mode(configuration)
    if reflection_mode == "not_recorded":
        notes.append("Beam-center mask reflection mode was not recorded or is unreadable")

    sample_id = _text(sample.get("sample_id") or scan_set.get("sample_id"))
    sample_name = _text(sample.get("sample_name"))
    polymer = _text(sample.get("polymer"))
    wash_state = _text(sample.get("wash_state"))
    details = [polymer]
    diameter = _number_label(sample.get("diameter_nm"))
    if diameter:
        details.append(f"{diameter} nm")
    details.append(wash_state)
    description = " | ".join(part for part in (
        sample_name or sample_id or scan_set_id,
        ", ".join(part for part in details if part),
        " ".join(part for part in (scan_type, region) if part),
    ) if part)
    scans = configuration.get("scans", [])
    pairs = configuration.get("polarization_pairs", [])
    if not isinstance(scans, list) or any(not isinstance(scan, dict) for scan in scans):
        raise ValueError("Scan metadata must be a list of dictionaries")
    if not isinstance(pairs, list):
        raise ValueError("Polarization-pair metadata must be a list")

    return {
        "sample_description": description,
        "sample_id": sample_id or None,
        "sample_name": sample_name or None,
        "polymer": polymer or None,
        "diameter_nm": sample.get("diameter_nm"),
        "wash_state": wash_state or None,
        **{name: sample.get(name) for name in ("np_vendor", "mw_g_mol", "mn_g_mol", "pdi")},
        "scan_set_id": scan_set_id,
        "scan_type": scan_type or None,
        "energy_region": region or None,
        "scan_type_recorded": recorded_type or None,
        "beam_center_reflection_mode": reflection_mode,
        **_export_review_metadata(manifest_path, manifest),
        "saved_at_utc": saved_at,
        "run_id": manifest_path.parent.parent.name,
        "is_latest": False,
        "scan_ids": ", ".join(_number_label(scan.get("scan_id")) for scan in scans),
        "polarizations": ", ".join(
            f"{_number_label(scan['polarization_deg'])}°"
            for scan in scans if scan.get("polarization_deg") is not None
        ),
        "scan_count": len(scans),
        "pair_count": len(pairs),
        "group_count": len(groups),
        "has_angular_fit": any(name.endswith("/angular_anisotropy_fit") for name in groups),
        "schema_version": manifest.get("schema_version"),
        "git_commit": manifest.get("git_commit"),
        "git_working_tree_clean": manifest.get("git_working_tree_clean"),
        "export_directory": str(manifest_path.parent.resolve()),
        "manifest_path": str(manifest_path.resolve()),
        "zarr_store": str(store),
        "metadata_notes": "; ".join(notes),
    }


def build_results_catalog(results_root=None):
    """Return one row per saved export, including every historical version.

    Read only manifest JSON and check for finalized store metadata files;
    do not open arrays or walk Zarr chunk directories. This is discovery,
    not a replacement for the workup's full export-verification cell.

    Missing, malformed, and unfinished exports are skipped and reported in
    catalog.attrs['skipped_exports'], with a summary warning. Metadata comes
    from each saved snapshot, not from the current configuration CSV files.
    """
    root = Path(results_root) if results_root is not None else RESULTS_BASE_DIR / "results"
    if not root.is_absolute():
        root = RESULTS_BASE_DIR / root
    root = root.resolve()
    if not root.is_dir():
        raise FileNotFoundError(
            f"Results directory does not exist: {root}. "
            "Set RESULTS_ROOT to the folder containing your saved exports, "
            "then rerun the refresh cell."
        )
    if root.name.endswith((".zarr", ".incomplete")):
        raise ValueError("Select a results directory, not a Zarr store or incomplete export")
    rows = []
    issues = []
    for path in _manifest_paths(root, issues):
        try:
            rows.append(_catalog_row(path))
        except (OSError, ValueError, TypeError) as error:
            issues.append({"manifest_path": str(path), "reason": str(error)})
    catalog = pd.DataFrame(rows, columns=CATALOG_COLUMNS)
    catalog["saved_at_utc"] = pd.to_datetime(catalog["saved_at_utc"], utc=True)
    for name in ("is_latest", "has_angular_fit"):
        catalog[name] = catalog[name].astype(bool)
    if not catalog.empty:
        # Manifest path makes equal-timestamp ties deterministic. This does
        # not imply that either export is scientifically preferable.
        catalog = catalog.sort_values(
            ["saved_at_utc", "manifest_path"], ascending=[False, True], kind="stable",
        ).reset_index(drop=True)
        catalog["is_latest"] = ~catalog["scan_set_id"].duplicated()
    catalog.attrs = {"results_root": str(root), "skipped_exports": issues}
    if issues:
        warnings.warn(
            f"Skipped {len(issues)} unavailable or malformed export locations. "
            "See catalog.attrs['skipped_exports'] for paths and reasons.",
            UserWarning, stacklevel=2,
        )
    return catalog


def select_results(
    catalog, *, latest_only=True, sample_id=None, sample_name=None,
    scan_type=None, energy_region=None, polymer=None, wash_state=None, search=None,
    reflection_mode=None,
    review_status=None,
):
    """Filter a catalog without modifying it or deleting saved versions.

    Named metadata filters are case-insensitive exact matches. Each accepts
    a single value or a list/tuple/set of values. Search is a literal,
    case-insensitive substring over the description and sample/scan-set IDs.
    Review status is selected BEFORE latest: an approved export is not
    hidden by a newer draft. Latest is then determined within that status
    selection BEFORE the other filters (including reflection mode).
    reflection_mode=None includes all settings; the string "none" selects
    no reflection and "not_recorded" selects missing/unreadable metadata.
    """
    if not isinstance(catalog, pd.DataFrame):
        raise TypeError("catalog must be a pandas DataFrame")
    if not isinstance(latest_only, bool):
        raise ValueError("latest_only must be True or False")
    required = set(CATALOG_COLUMNS)
    if not required.issubset(catalog.columns):
        raise ValueError("Use the DataFrame returned by build_results_catalog")
    selected = catalog.copy()
    if review_status is not None:
        values = review_status if isinstance(review_status, (list, tuple, set)) else [review_status]
        if any(isinstance(value, (dict, list, tuple, set)) for value in values):
            raise ValueError("review_status: use one value or a flat list of values")
        accepted = {_text(value).casefold() for value in values}
        selected = selected.loc[selected["review_status"].isin(accepted)]
    if latest_only:
        selected = selected.sort_values(
            ["saved_at_utc", "manifest_path"], ascending=[False, True], kind="stable",
        ).drop_duplicates("scan_set_id", keep="first")
    for name, criterion in (
        ("sample_id", sample_id), ("sample_name", sample_name),
        ("scan_type", scan_type), ("energy_region", energy_region),
        ("polymer", polymer), ("wash_state", wash_state),
        ("beam_center_reflection_mode", reflection_mode),
    ):
        if criterion is None:
            continue
        values = criterion if isinstance(criterion, (list, tuple, set)) else [criterion]
        if any(isinstance(value, (dict, list, tuple, set)) for value in values):
            raise ValueError(f"{name}: use one value or a flat list of values")
        accepted = {_text(value).casefold() for value in values}
        mask = selected[name].fillna("").astype(str).str.casefold().isin(accepted)
        selected = selected.loc[mask]
    if search is not None:
        if not isinstance(search, str):
            raise TypeError("search must be a string or None")
        needle = search.strip().casefold()
        mask = pd.Series(False, index=selected.index)
        for name in ("sample_description", "sample_id", "sample_name", "scan_set_id"):
            mask |= selected[name].fillna("").astype(str).str.casefold().str.contains(needle, regex=False)
        selected = selected.loc[mask]
    selected = selected.sort_values(
        ["sample_description", "scan_set_id", "saved_at_utc", "manifest_path"],
        ascending=[True, True, False, True], kind="stable",
    ).reset_index(drop=True)
    selected.attrs = dict(catalog.attrs)
    return selected


In [ ]:
# None finds a results folder in the working directory or its parents.
# On another computer, set an absolute path to your copied results folder.
# Example: RESULTS_ROOT = "/path/to/copied/results"
RESULTS_ROOT = None

catalog = build_results_catalog(RESULTS_ROOT)
for stale_name in ("view", "selected_export", "selected_manifest", "group_table"):
    globals().pop(stale_name, None)

print("Results folder:", catalog.attrs["results_root"])
print(f"Saved exports: {len(catalog):,}")
print(f"Scan sets with saved results: {catalog['scan_set_id'].nunique():,}")

print("Review status counts:", catalog["review_status"].value_counts().to_dict())

skipped = catalog.attrs["skipped_exports"]
if skipped:
    print("These locations were skipped; no files were changed:")
    display(pd.DataFrame(skipped))
elif catalog.empty:
    print("No saved exports found in this folder.")


## Find results

Leave a filter as `None` to include all values. Named filters match exactly,
ignoring capitalization. They may also be lists, such as `ENERGY_REGION = ["C", "O"]`.
`SEARCH` looks for a literal fragment of a sample description, sample ID, or scan-set ID.

For example, use `SCAN_TYPE = "long"` and `ENERGY_REGION = "O"` for long oxygen scans,
or `SAMPLE_ID = "ksr216"` for one sample.
Set `REVIEW_STATUS = "approved"` for reviewed results, `"draft"` for new attempts,
`"development"` for old workflow-development exports, or `None` for every status.
`"unknown"` means a review record needs inspection; it is excluded from approved results.
A list can select several statuses.

Review status is selected first, then the newest export within that selection if
`LATEST_ONLY = True`, then all other filters. Thus a newer draft cannot hide an
approved result, but reflection filtering does not substitute an older approved export.

After a new workup is saved, rerun the refresh cell above and this cell.
The table displays at most `DISPLAY_LIMIT` rows; narrow the filters or increase that limit.
### Compare beam-center mask reflection settings

The table's `beam-center mask reflection` column comes from the saved manifest's
`configuration.processing.beam_center_fit.reflection_mode`, not today's TOML.
`REFLECTION_MODE = None` includes all settings. Use a string to filter:
`"none"` (no reflection), `"twofold"`, `"fourfold"`, or `"eightfold"`.
A list such as `["none", "fourfold"]` includes both settings.
`"not_recorded"` selects older exports without readable reflection metadata;
missing metadata is never assumed to mean `"none"` or `"fourfold"`.

Keep `LATEST_ONLY = False` when comparing saved runs with different settings for
one scan set. If `True`, only the newest export per scan set within the chosen review status is considered,
before reflection filtering; older matching exports are not substituted.

This is a saved beam-centering setting, not a claim that center fitting was run.
It may also be recorded when the configured center was used without fitting.
It does not describe a reflection applied to the final intensity/anisotropy data.



In [ ]:
for stale_name in ("selected_export", "selected_manifest", "group_table"):
    globals().pop(stale_name, None)

REVIEW_STATUS = None  # None = all; "draft", "development", "approved",or "unknown".
LATEST_ONLY = True
SAMPLE_ID = None
SAMPLE_NAME = None
SCAN_TYPE = None
ENERGY_REGION = None
REFLECTION_MODE = None  # None = all modes; "none" = no mask reflection.
POLYMER = None
WASH_STATE = None
SEARCH = None
DISPLAY_LIMIT = 100

view = select_results(
    catalog,
    latest_only=LATEST_ONLY,
    review_status=REVIEW_STATUS,
    sample_id=SAMPLE_ID,
    sample_name=SAMPLE_NAME,
    scan_type=SCAN_TYPE,
    energy_region=ENERGY_REGION,
    reflection_mode=REFLECTION_MODE,
    polymer=POLYMER,
    wash_state=WASH_STATE,
    search=SEARCH,
)

if isinstance(DISPLAY_LIMIT, bool) or not isinstance(DISPLAY_LIMIT, int) or DISPLAY_LIMIT < 1:
    raise ValueError("DISPLAY_LIMIT must be a positive integer")
print(f"Matching exports: {len(view):,}; showing {min(len(view), DISPLAY_LIMIT):,}")
print("Review filter:", REVIEW_STATUS if REVIEW_STATUS is not None else "all statuses")
if view.empty:
    print("No matching reviewed results. To inspect drafts or the development archive, change REVIEW_STATUS (and RESULTS_ROOT for the archive).")
print("View:", "latest save per scan set within the review filter" if LATEST_ONLY else "all saved versions")

columns = [
    "sample_description", "scan_set_id", "saved_at_utc",
    "review_status", "reviewer", "beam_center_reflection_mode",
    "scan_ids", "group_count", "is_latest", "metadata_notes",
]
with pd.option_context("display.max_rows", DISPLAY_LIMIT, "display.max_colwidth", None):
    display(view.head(DISPLAY_LIMIT)[columns].rename(columns={
        "beam_center_reflection_mode": "beam-center mask reflection",
    }))


In [ ]:
### Cell 1 — function

def plot_saved_anisotropy_reviews(
    displayed_exports,
    *,
    normalization_mode="half_ring",
    color_limits=None,
    full_energy_ranges_ev=None,
    zoom_ranges_ev=None,
    energy_markers_ev=None,
    trace_xlim=None,
    trace_ylim=None,
    map_panel_size=(7, 5),
    trace_figsize=(15, 9),
    cmap="coolwarm",
    style="dark_background",
    legend_fontsize=8,
    trace_linewidth=1.5,
):
    """Plot four map figures and one trace figure from saved χ-averaged A."""
    import json
    import re
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize
    import xarray as xr

    if color_limits is None:
        color_limits = {
            "C_full": (-0.15, 0.15),
            "O_full": (-0.15, 0.15),
            "C_zoom": (-0.15, 0.15),
            "O_zoom": (-0.15, 0.15),
        }
    if full_energy_ranges_ev is None:
        full_energy_ranges_ev = {"C": None, "O": None}
    if zoom_ranges_ev is None:
        zoom_ranges_ev = {"C": (282, 292), "O": (538, 548)}
    if energy_markers_ev is None:
        energy_markers_ev = {
            "C": [284.8, 286.0, 289],
            "O": [533.0, 535.0],
        }

    def checked_limits(bounds):
        if bounds is None:
            return None
        bounds = np.asarray(bounds, dtype=float)
        if (
            bounds.shape != (2,)
            or not np.all(np.isfinite(bounds))
            or bounds[0] >= bounds[1]
        ):
            raise ValueError(
                "Limits must be two finite numbers with lower < upper."
            )
        return tuple(bounds)

    def trace_limits(setting, family, region):
        # Accept common limits, regional limits, or individual-panel limits.
        if isinstance(setting, dict):
            setting = setting.get((family, region), setting.get(region))
        return checked_limits(setting)

    for key in ("C_full", "O_full", "C_zoom", "O_zoom"):
        checked_limits(color_limits[key])

    # Load each displayed export/pair once.
    panels = []
    for _, row in displayed_exports.iterrows():
        with Path(row["manifest_path"]).open(encoding="utf-8") as file:
            manifest = json.load(file)

        groups = [
            group for group in manifest["zarr_groups"]
            if group.endswith("/chi_averaged_pointwise_anisotropy")
        ]
        if not groups:
            print(
                "No saved χ-averaged pointwise result:",
                row["scan_set_id"],
            )

        sample_match = re.search(
            r"(?:^|_)(EB[12])([A-Z])(?=_|$)",
            str(row["sample_id"]).strip().upper(),
        )
        family = sample_match.group(1) if sample_match else None
        letter = sample_match.group(2) if sample_match else None
        region = str(row["energy_region"]).strip().upper()
        is_long = str(row["scan_type"]).strip().casefold() == "long"

        for group in sorted(groups):
            with xr.open_zarr(
                row["zarr_store"],
                group=group,
                consolidated=True,
                chunks=None,
            ) as dataset:
                values = (
                    dataset["anisotropy"]
                    .sel(normalization_mode=normalization_mode)
                    .transpose("energy", "q")
                    .sortby("energy")
                    .load()
                )

            panels.append({
                "row": row,
                "group": group,
                "pair": group.split("/")[-2].split("_pair_")[-1],
                "values": values,
                "family": family,
                "letter": letter,
                "region": region,
                "is_long": is_long,
            })

    if not panels:
        raise ValueError(
            "No χ-averaged pointwise maps in the displayed exports."
        )

    grouped = {
        (family, region): []
        for family in ("EB1", "EB2")
        for region in ("C", "O")
    }

    for panel in panels:
        key = (panel["family"], panel["region"])
        if (
            panel["is_long"]
            and panel["letter"] in ("A", "B", "C")
            and key in grouped
        ):
            grouped[key].append(panel)

    for entries in grouped.values():
        entries.sort(
            key=lambda panel: (
                panel["letter"],
                str(panel["row"]["scan_set_id"]),
                str(panel["row"]["saved_at_utc"]),
                panel["group"],
            )
        )

    with plt.style.context(style):
        # Figures 1–4: C full, O full, C zoom, O zoom.
        for region, zoomed in (
            ("C", False),
            ("O", False),
            ("C", True),
            ("O", True),
        ):
            family_rows = [
                grouped[("EB1", region)],
                grouped[("EB2", region)],
            ]
            ncols = max(len(entries) for entries in family_rows)

            if ncols == 0:
                print(f"No long {region} maps; skipping this figure.")
                continue

            color_key = f"{region}_{'zoom' if zoomed else 'full'}"
            norm = Normalize(*checked_limits(color_limits[color_key]))
            energy_range = checked_limits(
                zoom_ranges_ev.get(region)
                if zoomed
                else full_energy_ranges_ev.get(region)
            )

            fig, axes = plt.subplots(
                2, ncols,
                figsize=(
                    map_panel_size[0] * ncols,
                    map_panel_size[1] * 2,
                ),
                squeeze=False,
                constrained_layout=True,
            )
            active_axes = []

            for row_index, family in enumerate(("EB1", "EB2")):
                entries = grouped[(family, region)]

                for column_index, panel in enumerate(entries):
                    ax = axes[row_index, column_index]
                    active_axes.append(ax)
                    row = panel["row"]
                    values = panel["values"]

                    image = ax.pcolormesh(
                        values.q,
                        values.energy,
                        np.ma.masked_invalid(values.values),
                        shading="auto",
                        cmap=cmap,
                        norm=norm,
                    )

                    if energy_range is not None:
                        ax.set_ylim(*energy_range)
                        energies = values.energy.values
                        if not np.any(
                            (energies >= energy_range[0])
                            & (energies <= energy_range[1])
                        ):
                            ax.text(
                                0.5, 0.5,
                                "No measured energies in this window",
                                transform=ax.transAxes,
                                ha="center", va="center",
                            )

                    if zoomed:
                        for energy in energy_markers_ev.get(region, []):
                            if (
                                energy_range is None
                                or energy_range[0] <= energy <= energy_range[1]
                            ):
                                ax.axhline(
                                    energy,
                                    color="white",
                                    linestyle=":",
                                    linewidth=1,
                                    label=f"{energy:g} eV",
                                )
                        if ax.lines:
                            ax.legend(
                                fontsize=legend_fontsize,
                                loc="upper right",
                            )

                    ax.set(
                        xlabel="q (Å⁻¹)",
                        ylabel=(
                            f"{family}\nEnergy (eV)"
                            if column_index == 0
                            else "Energy (eV)"
                        ),
                        title=(
                            f"{row['sample_description']}\n"
                            f"{row['scan_set_id']}; pair {panel['pair']}"
                        ),
                    )

                for unused_ax in axes[row_index, len(entries):]:
                    unused_ax.set_visible(False)

            fig.colorbar(
                image,
                ax=active_axes,
                label=f"χ-averaged pointwise A ({normalization_mode})",
                shrink=0.85,
            )
            region_name = "carbon" if region == "C" else "oxygen"
            range_label = (
                f"{energy_range[0]:g}–{energy_range[1]:g} eV"
                if energy_range is not None
                else "full energy range"
            )
            fig.suptitle(f"Long {region_name} scans — {range_label}")
            plt.show()
            plt.close(fig)

        # Final figure: EB1 above EB2; carbon left, oxygen right.
        # Dictionary limits allow independently controlled panels.
        fig, trace_axes = plt.subplots(
            2, 2,
            figsize=trace_figsize,
            sharex=False if isinstance(trace_xlim, dict) else "col",
            sharey=False if isinstance(trace_ylim, dict) else True,
            constrained_layout=True,
        )

        colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        line_styles = ["-", "--", "-.", ":"]

        for family_index, family in enumerate(("EB1", "EB2")):
            for region_index, region in enumerate(("C", "O")):
                ax = trace_axes[family_index, region_index]

                for panel_index, panel in enumerate(panels):
                    if (
                        not panel["is_long"]
                        or panel["region"] != region
                        or panel["family"] != family
                    ):
                        continue

                    row = panel["row"]
                    pair = panel["pair"]
                    values = panel["values"]
                    available = values.energy.values

                    for energy_index, requested in enumerate(
                        energy_markers_ev.get(region, [])
                    ):
                        if not available.min() <= requested <= available.max():
                            print(
                                f"{row['scan_set_id']}; pair {pair}: "
                                f"skipping {requested:g} eV "
                                "(outside the saved energy range)"
                            )
                            continue

                        trace = values.sel(energy=requested, method="nearest")
                        selected = float(trace.energy.values)

                        if not np.isfinite(trace.values).any():
                            print(
                                f"{row['scan_set_id']}; pair {pair}: "
                                f"no finite anisotropy at {selected:g} eV"
                            )
                            continue

                        energy_label = f"{selected:g} eV"
                        if not np.isclose(
                            selected, requested, rtol=0, atol=1e-6
                        ):
                            energy_label += f" (requested {requested:g})"
                            print(
                                f"{row['scan_set_id']}; pair {pair}: "
                                f"requested {requested:g} eV "
                                f"→ selected {selected:g} eV"
                            )

                        ax.plot(
                            trace.q,
                            trace,
                            color=colors[energy_index % len(colors)],
                            linestyle=line_styles[
                                panel_index % len(line_styles)
                            ],
                            linewidth=trace_linewidth,
                            label=(
                                f"{row['sample_description']}; "
                                f"pair {pair}; {energy_label}"
                            ),
                        )

                if ax.lines:
                    ax.legend(fontsize=legend_fontsize, loc="best")
                else:
                    ax.text(
                        0.5, 0.5,
                        "No available traces for the selected energies",
                        transform=ax.transAxes,
                        ha="center", va="center",
                    )

                ax.axhline(
                    0, color="gray", linestyle=":", linewidth=0.8
                )
                ax.set(
                    xlabel="q (Å⁻¹)",
                    ylabel="χ-averaged pointwise A",
                    title=(
                        f"{family} — long carbon scans"
                        if region == "C"
                        else f"{family} — long oxygen scans"
                    ),
                )

                xlimits = trace_limits(trace_xlim, family, region)
                ylimits = trace_limits(trace_ylim, family, region)
                if xlimits is not None:
                    ax.set_xlim(*xlimits)
                if ylimits is not None:
                    ax.set_ylim(*ylimits)

        fig.suptitle(
            "Selected-energy χ-averaged pointwise anisotropy; "
            f"{normalization_mode}"
        )
        plt.show()
        plt.close(fig)


In [ ]:
### Cell 2 — settings and plotting

NORMALIZATION_MODE = "half_ring"

# Independent shared colorbar limits for each map figure.
COLOR_LIMITS = {
    "C_full": (-0.15, 0.15),
    "O_full": (-0.15, 0.15),
    "C_zoom": (-0.05, 0.05),
    "O_zoom": (-0.15, 0.15),
}

# None leaves the original maps at their full energy range.
FULL_ENERGY_RANGES_EV = {
    "C": None,
    "O": None,
}

ZOOM_RANGES_EV = {
    "C": (282, 292),
    "O": (528, 538),
}

ENERGY_MARKERS_EV = {
    "C": [284.8, 286.0, 289],
    "O": [533.0, 535.0],
}

# Final trace figure: None = automatic; a tuple applies to all panels.
TRACE_XLIM = None  # Example: (0.005, 0.06)
TRACE_YLIM = None  # Example: (-0.15, 0.15)

# Alternatively, specify limits by region:
# TRACE_XLIM = {"C": (0.005, 0.06), "O": (0.005, 0.10)}
#
# Or independently for each panel:
# TRACE_YLIM = {
#     ("EB1", "C"): (-0.10, 0.10),
#     ("EB2", "C"): (-0.15, 0.15),
#     ("EB1", "O"): (-0.10, 0.10),
#     ("EB2", "O"): (-0.15, 0.15),
# }

MAP_PANEL_SIZE = (7, 5)   # Width and height per map subplot.
TRACE_FIGSIZE = (15, 9)
CMAP = "coolwarm"
STYLE = "dark_background"
LEGEND_FONTSIZE = 8
TRACE_LINEWIDTH = 1.5

plot_saved_anisotropy_reviews(
    view.head(DISPLAY_LIMIT),
    normalization_mode=NORMALIZATION_MODE,
    color_limits=COLOR_LIMITS,
    full_energy_ranges_ev=FULL_ENERGY_RANGES_EV,
    zoom_ranges_ev=ZOOM_RANGES_EV,
    energy_markers_ev=ENERGY_MARKERS_EV,
    trace_xlim=TRACE_XLIM,
    trace_ylim=TRACE_YLIM,
    map_panel_size=MAP_PANEL_SIZE,
    trace_figsize=TRACE_FIGSIZE,
    cmap=CMAP,
    style=STYLE,
    legend_fontsize=LEGEND_FONTSIZE,
    trace_linewidth=TRACE_LINEWIDTH,
)


In [ ]:
### Cell 1 — new function

def plot_saved_anisotropy_window_reviews(
    displayed_exports,
    *,
    normalization_mode="half_ring",
    color_limits=None,
    full_energy_ranges_ev=None,
    zoom_ranges_ev=None,
    energy_markers_ev=None,
    average_energy=True,
    energy_halfwidths_ev=None,
    energy_halfwidth_overrides_ev=None,
    require_all_energies=True,
    trace_xlim=None,
    trace_ylim=None,
    map_panel_size=(7, 5),
    trace_figsize=(15, 9),
    cmap="coolwarm",
    style="dark_background",
    legend_fontsize=8,
    trace_linewidth=1.5,
):
    """Plot saved χ-averaged A maps and signed energy-window mean traces."""
    import json
    import re
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize
    import xarray as xr

    if color_limits is None:
        color_limits = {
            "C_full": (-0.15, 0.15),
            "O_full": (-0.15, 0.15),
            "C_zoom": (-0.15, 0.15),
            "O_zoom": (-0.15, 0.15),
        }
    if full_energy_ranges_ev is None:
        full_energy_ranges_ev = {"C": None, "O": None}
    if zoom_ranges_ev is None:
        zoom_ranges_ev = {"C": (282, 292), "O": (528, 538)}
    if energy_markers_ev is None:
        energy_markers_ev = {
            "C": [284.8, 286.0, 289.0],
            "O": [533.0, 535.0],
        }
    if energy_halfwidths_ev is None:
        energy_halfwidths_ev = {"C": 0.20, "O": 0.40}
    if energy_halfwidth_overrides_ev is None:
        energy_halfwidth_overrides_ev = {"C": {289.0: 0.50}, "O": {}}

    def checked_limits(bounds):
        if bounds is None:
            return None
        bounds = np.asarray(bounds, dtype=float)
        if (
            bounds.shape != (2,)
            or not np.all(np.isfinite(bounds))
            or bounds[0] >= bounds[1]
        ):
            raise ValueError(
                "Limits must be two finite numbers with lower < upper."
            )
        return tuple(bounds)

    def trace_limits(setting, family, region):
        if isinstance(setting, dict):
            setting = setting.get((family, region), setting.get(region))
        return checked_limits(setting)

    for key in ("C_full", "O_full", "C_zoom", "O_zoom"):
        checked_limits(color_limits[key])

    for region in ("C", "O"):
        for energy in energy_markers_ev.get(region, []):
            if not np.isfinite(energy) or energy <= 0:
                raise ValueError("Selected energies must be positive and finite.")

    def make_trace(values, region, requested):
        """Average existing A values, not intensities or fitted curves."""
        available = np.asarray(values.energy.values, dtype=float)
        info = {
            "requested_energy_ev": float(requested),
            "mode": "window_mean" if average_energy else "single",
            "energy_count": 0,
        }
        tolerance = 1e-8

        if average_energy:
            halfwidth = float(
                energy_halfwidth_overrides_ev.get(region, {}).get(
                    requested, energy_halfwidths_ev[region]
                )
            )
            if not np.isfinite(halfwidth) or halfwidth < 0:
                raise ValueError("Energy halfwidths must be finite and nonnegative.")

            lower = requested - halfwidth
            upper = requested + halfwidth
            info["halfwidth_ev"] = halfwidth

            # Do not silently truncate a window at the saved grid's edges.
            if (
                lower < available.min() - tolerance
                or upper > available.max() + tolerance
            ):
                info["status"] = "Skipped: window extends beyond saved range"
                return None, info

            block = values.sel(
                energy=slice(lower - tolerance, upper + tolerance)
            )
        else:
            info["halfwidth_ev"] = 0.0
            if not available.min() <= requested <= available.max():
                info["status"] = "Skipped: energy outside saved range"
                return None, info

            index = int(np.argmin(np.abs(available - requested)))
            block = values.isel(energy=[index])

        count = block.sizes["energy"]
        info["energy_count"] = count
        if count == 0:
            info["status"] = "Skipped: no measured energies in window"
            return None, info

        actual_energies = np.asarray(block.energy.values, dtype=float)
        info.update({
            "energy_min_ev": float(actual_energies.min()),
            "energy_max_ev": float(actual_energies.max()),
            "selected_energies_ev": actual_energies.tolist(),
        })

        finite = np.isfinite(block)
        contributing_count = finite.sum("energy")

        # Equal-weight, signed mean of the existing χ-averaged anisotropy.
        trace = block.where(finite).mean("energy", skipna=True)

        if require_all_energies:
            trace = trace.where(contributing_count == count)

        valid_q = np.isfinite(trace.values)
        info["valid_q_count"] = int(valid_q.sum())

        if not valid_q.any():
            info["status"] = "Skipped: no valid q points"
            return None, info

        counts = contributing_count.values[valid_q]
        info["contributors_per_q_min"] = int(counts.min())
        info["contributors_per_q_max"] = int(counts.max())
        info["status"] = "Plotted"

        return trace, info

    # Load each displayed export/pair once.
    panels = []
    for _, row in displayed_exports.iterrows():
        with Path(row["manifest_path"]).open(encoding="utf-8") as file:
            manifest = json.load(file)

        groups = [
            group for group in manifest["zarr_groups"]
            if group.endswith("/chi_averaged_pointwise_anisotropy")
        ]
        if not groups:
            print(
                "No saved χ-averaged pointwise result:",
                row["scan_set_id"],
            )

        sample_match = re.search(
            r"(?:^|_)(EB[12])([A-Z])(?=_|$)",
            str(row["sample_id"]).strip().upper(),
        )
        family = sample_match.group(1) if sample_match else None
        letter = sample_match.group(2) if sample_match else None
        region = str(row["energy_region"]).strip().upper()
        is_long = str(row["scan_type"]).strip().casefold() == "long"

        for group in sorted(groups):
            with xr.open_zarr(
                row["zarr_store"],
                group=group,
                consolidated=True,
                chunks=None,
            ) as dataset:
                values = (
                    dataset["anisotropy"]
                    .sel(normalization_mode=normalization_mode)
                    .transpose("energy", "q")
                    .sortby("energy")
                    .load()
                )

            panels.append({
                "row": row,
                "group": group,
                "pair": group.split("/")[-2].split("_pair_")[-1],
                "values": values,
                "family": family,
                "letter": letter,
                "region": region,
                "is_long": is_long,
            })

    if not panels:
        raise ValueError(
            "No χ-averaged pointwise maps in the displayed exports."
        )

    grouped = {
        (family, region): []
        for family in ("EB1", "EB2")
        for region in ("C", "O")
    }

    for panel in panels:
        key = (panel["family"], panel["region"])
        if (
            panel["is_long"]
            and panel["letter"] in ("A", "B", "C")
            and key in grouped
        ):
            grouped[key].append(panel)

    for entries in grouped.values():
        entries.sort(
            key=lambda panel: (
                panel["letter"],
                str(panel["row"]["scan_set_id"]),
                str(panel["row"]["saved_at_utc"]),
                panel["group"],
            )
        )

    trace_report = []

    with plt.style.context(style):
        # Figures 1–4: unchanged, unsmoothed maps.
        for region, zoomed in (
            ("C", False),
            ("O", False),
            ("C", True),
            ("O", True),
        ):
            family_rows = [
                grouped[("EB1", region)],
                grouped[("EB2", region)],
            ]
            ncols = max(len(entries) for entries in family_rows)

            if ncols == 0:
                print(f"No long {region} maps; skipping this figure.")
                continue

            color_key = f"{region}_{'zoom' if zoomed else 'full'}"
            norm = Normalize(*checked_limits(color_limits[color_key]))
            energy_range = checked_limits(
                zoom_ranges_ev.get(region)
                if zoomed
                else full_energy_ranges_ev.get(region)
            )

            fig, axes = plt.subplots(
                2, ncols,
                figsize=(
                    map_panel_size[0] * ncols,
                    map_panel_size[1] * 2,
                ),
                squeeze=False,
                constrained_layout=True,
            )
            active_axes = []

            for row_index, family in enumerate(("EB1", "EB2")):
                entries = grouped[(family, region)]

                for column_index, panel in enumerate(entries):
                    ax = axes[row_index, column_index]
                    active_axes.append(ax)
                    row = panel["row"]
                    values = panel["values"]

                    image = ax.pcolormesh(
                        values.q,
                        values.energy,
                        np.ma.masked_invalid(values.values),
                        shading="auto",
                        cmap=cmap,
                        norm=norm,
                    )

                    if energy_range is not None:
                        ax.set_ylim(*energy_range)
                        energies = values.energy.values
                        if not np.any(
                            (energies >= energy_range[0])
                            & (energies <= energy_range[1])
                        ):
                            ax.text(
                                0.5, 0.5,
                                "No measured energies in this window",
                                transform=ax.transAxes,
                                ha="center", va="center",
                            )

                    if zoomed:
                        for energy in energy_markers_ev.get(region, []):
                            if (
                                energy_range is None
                                or energy_range[0] <= energy <= energy_range[1]
                            ):
                                ax.axhline(
                                    energy,
                                    color="white",
                                    linestyle=":",
                                    linewidth=1,
                                    label=f"{energy:g} eV",
                                )
                        if ax.lines:
                            ax.legend(
                                fontsize=legend_fontsize,
                                loc="upper right",
                            )

                    ax.set(
                        xlabel="q (Å⁻¹)",
                        ylabel=(
                            f"{family}\nEnergy (eV)"
                            if column_index == 0
                            else "Energy (eV)"
                        ),
                        title=(
                            f"{row['sample_description']}\n"
                            f"{row['scan_set_id']}; pair {panel['pair']}"
                        ),
                    )

                for unused_ax in axes[row_index, len(entries):]:
                    unused_ax.set_visible(False)

            fig.colorbar(
                image,
                ax=active_axes,
                label=f"χ-averaged pointwise A ({normalization_mode})",
                shrink=0.85,
            )
            region_name = "carbon" if region == "C" else "oxygen"
            range_label = (
                f"{energy_range[0]:g}–{energy_range[1]:g} eV"
                if energy_range is not None
                else "full energy range"
            )
            fig.suptitle(f"Long {region_name} scans — {range_label}")
            plt.show()
            plt.close(fig)

        # Final figure: energy-window means or single-energy traces.
        fig, trace_axes = plt.subplots(
            2, 2,
            figsize=trace_figsize,
            sharex=False if isinstance(trace_xlim, dict) else "col",
            sharey=False if isinstance(trace_ylim, dict) else True,
            constrained_layout=True,
        )

        colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        line_styles = ["-", "--", "-.", ":"]

        for family_index, family in enumerate(("EB1", "EB2")):
            for region_index, region in enumerate(("C", "O")):
                ax = trace_axes[family_index, region_index]

                for panel_index, panel in enumerate(panels):
                    if (
                        not panel["is_long"]
                        or panel["region"] != region
                        or panel["family"] != family
                    ):
                        continue

                    row = panel["row"]
                    pair = panel["pair"]

                    for energy_index, requested in enumerate(
                        energy_markers_ev.get(region, [])
                    ):
                        trace, info = make_trace(
                            panel["values"], region, requested
                        )
                        trace_report.append({
                            "sample_id": row["sample_id"],
                            "scan_set_id": row["scan_set_id"],
                            "pair": pair,
                            "region": region,
                            **info,
                        })

                        if trace is None:
                            continue

                        count = info["energy_count"]
                        lower = info["energy_min_ev"]
                        upper = info["energy_max_ev"]

                        if average_energy:
                            energy_label = (
                                f"{lower:g}–{upper:g} eV; n={count}"
                            )
                        else:
                            energy_label = f"{lower:g} eV"
                            if not np.isclose(
                                lower, requested, rtol=0, atol=1e-6
                            ):
                                energy_label += (
                                    f" (requested {requested:g})"
                                )

                        ax.plot(
                            trace.q,
                            trace,
                            color=colors[energy_index % len(colors)],
                            linestyle=line_styles[
                                panel_index % len(line_styles)
                            ],
                            linewidth=trace_linewidth,
                            label=(
                                f"{row['sample_description']}; "
                                f"pair {pair}; {energy_label}"
                            ),
                        )

                if ax.lines:
                    ax.legend(fontsize=legend_fontsize, loc="best")
                else:
                    ax.text(
                        0.5, 0.5,
                        "No available traces for the selected windows",
                        transform=ax.transAxes,
                        ha="center", va="center",
                    )

                ax.axhline(
                    0, color="gray", linestyle=":", linewidth=0.8
                )
                ax.set(
                    xlabel="q (Å⁻¹)",
                    ylabel=(
                        "Energy-window mean of χ-averaged A"
                        if average_energy
                        else "χ-averaged pointwise A"
                    ),
                    title=(
                        f"{family} — long carbon scans"
                        if region == "C"
                        else f"{family} — long oxygen scans"
                    ),
                )

                xlimits = trace_limits(trace_xlim, family, region)
                ylimits = trace_limits(trace_ylim, family, region)
                if xlimits is not None:
                    ax.set_xlim(*xlimits)
                if ylimits is not None:
                    ax.set_ylim(*ylimits)

        fig.suptitle(
            (
                "Energy-window means of χ-averaged pointwise A"
                if average_energy
                else "Single-energy χ-averaged pointwise A"
            )
            + f"; {normalization_mode}"
        )
        plt.show()
        plt.close(fig)

    return pd.DataFrame(trace_report)



In [ ]:
### Cell 2 — settings and function call

NORMALIZATION_MODE = "half_ring"

# Independent color limits for each map figure.
COLOR_LIMITS = {
    "C_full": (-0.15, 0.15),
    "O_full": (-0.15, 0.15),
    "C_zoom": (-0.05, 0.05),
    "O_zoom": (-0.15, 0.15),
}

FULL_ENERGY_RANGES_EV = {
    "C": None,
    "O": None,
}

ZOOM_RANGES_EV = {
    "C": (282, 292),
    "O": (528, 538),
}

# Window centers, also shown as dotted lines on the zoomed maps.
ENERGY_MARKERS_EV = {
    "C": [284.8, 286.0, 289.0],
    "O": [533.0, 535.0],
}

# True = energy-window means; False = nearest single-energy traces.
AVERAGE_ENERGY = True

# Default window HALFwidths, in eV.
ENERGY_HALFWIDTHS_EV = {
    "C": 0.20,
    "O": 0.40,
}

# Override individual centers where the sampling is coarser.
ENERGY_HALFWIDTH_OVERRIDES_EV = {
    "C": {
        289.0: 0.50,
    },
    "O": {},
}

# True: show a q point only if every selected energy is valid there.
# False: average available finite values; contributor counts can vary with q.
REQUIRE_ALL_ENERGIES = True

TRACE_XLIM = None  # Example: (0.005, 0.06)
TRACE_YLIM = None  # Example: (-0.15, 0.15)

# Regional limits are also supported:
# TRACE_XLIM = {"C": (0.005, 0.06), "O": (0.005, 0.10)}
#
# Or individual-panel limits:
# TRACE_YLIM = {
#     ("EB1", "C"): (-0.10, 0.10),
#     ("EB2", "C"): (-0.15, 0.15),
#     ("EB1", "O"): (-0.10, 0.10),
#     ("EB2", "O"): (-0.15, 0.15),
# }

MAP_PANEL_SIZE = (7, 5)
TRACE_FIGSIZE = (15, 9)
CMAP = "coolwarm"
STYLE = "dark_background"
LEGEND_FONTSIZE = 8
TRACE_LINEWIDTH = 1.5

energy_window_report = plot_saved_anisotropy_window_reviews(
    view.head(DISPLAY_LIMIT),
    normalization_mode=NORMALIZATION_MODE,
    color_limits=COLOR_LIMITS,
    full_energy_ranges_ev=FULL_ENERGY_RANGES_EV,
    zoom_ranges_ev=ZOOM_RANGES_EV,
    energy_markers_ev=ENERGY_MARKERS_EV,
    average_energy=AVERAGE_ENERGY,
    energy_halfwidths_ev=ENERGY_HALFWIDTHS_EV,
    energy_halfwidth_overrides_ev=ENERGY_HALFWIDTH_OVERRIDES_EV,
    require_all_energies=REQUIRE_ALL_ENERGIES,
    trace_xlim=TRACE_XLIM,
    trace_ylim=TRACE_YLIM,
    map_panel_size=MAP_PANEL_SIZE,
    trace_figsize=TRACE_FIGSIZE,
    cmap=CMAP,
    style=STYLE,
    legend_fontsize=LEGEND_FONTSIZE,
    trace_linewidth=TRACE_LINEWIDTH,
)

# Actual energies, point counts, q coverage, and any skipped windows.
display(energy_window_report)

## Inspect one saved export

Set `EXPORT_ROW` to the numbered row at the left of the table.
This cell shows its exact saved location and exported groups, without loading arrays.
The row number is only a selector for the current filtered table; the paths and
`scan_set_id` are the identifiers to keep for later use.


In [ ]:
import json
from pathlib import Path

EXPORT_ROW = 0

if view.empty:
    raise ValueError("No exports match. Change the filters and rerun the table cell.")
if (
    isinstance(EXPORT_ROW, bool)
    or not isinstance(EXPORT_ROW, int)
    or not 0 <= EXPORT_ROW < len(view)
):
    raise ValueError(f"EXPORT_ROW must be between 0 and {len(view) - 1}")

selected_export = view.iloc[EXPORT_ROW].to_dict()
print("Sample:", selected_export["sample_description"])
print("Scan set:", selected_export["scan_set_id"])
print("Saved at (UTC):", selected_export["saved_at_utc"])
print("Review status:", selected_export["review_status"])
print("Reviewed by:", selected_export["reviewer"])
print("Reviewed at (UTC):", selected_export["reviewed_at_utc"])
print("Review note:", selected_export["review_note"])
print("Beam-center mask reflection (saved setting):",
      selected_export["beam_center_reflection_mode"])
print("Output directory:", selected_export["export_directory"])
print("Zarr store:", selected_export["zarr_store"])
print("Manifest:", selected_export["manifest_path"])
print("Git commit at save:", selected_export["git_commit"])
print("Working tree clean at save:", selected_export["git_working_tree_clean"])
if selected_export["metadata_notes"]:
    print("Metadata notes:", selected_export["metadata_notes"])

with Path(selected_export["manifest_path"]).open(encoding="utf-8") as file:
    selected_manifest = json.load(file)

group_table = pd.DataFrame([
    {"group": group, "variables": ", ".join(variables)}
    for group, variables in selected_manifest["zarr_groups"].items()
])
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(group_table)


## Open a saved dataset (optional, read-only)

The cells above read metadata only. To open arrays for later analysis, set
`OPEN_DATASET = True` and `GROUP_NAME` to an exact group from the table above.

This needs ordinary xarray, Zarr 2 (`zarr>=2.18,<3`), and numcodecs. It does not
need the workup package or its processing environment. `chunks=None` avoids a
Dask dependency. Opening displays the dataset without loading every image;
accessing values loads the requested data.

The cell does not write to the store. Each pair retains its own native q grid;
nothing is aligned or combined between pairs or scan sets. The notebook may be
extended for later analysis, but any writes you add would be your own operations.



In [ ]:
OPEN_DATASET = False
GROUP_NAME = None  # Copy an exact group name from the inspection table.

# Clear a previously opened selection before opening another one.
if "selected_dataset" in globals():
    selected_dataset.close()
    del selected_dataset

if OPEN_DATASET:
    if "selected_export" not in globals() or "selected_manifest" not in globals():
        raise ValueError("Run the export-inspection cell before opening a dataset.")
    available_groups = selected_manifest["zarr_groups"]
    if not isinstance(GROUP_NAME, str) or GROUP_NAME not in available_groups:
        raise ValueError(
            "Set GROUP_NAME to an exact exported group from the inspection table."
        )

    import xarray as xr

    selected_dataset = xr.open_zarr(
        selected_export["zarr_store"],
        group=GROUP_NAME,
        consolidated=True,
        chunks=None,
    )
    print("Opened read-only:", GROUP_NAME)
    display(selected_dataset)
else:
    print("Metadata-only browsing; no dataset opened.")


## Return to this saved version later

Keep the selected `scan_set_id`, export path, and save time.
A fresh catalog may mark a newer export as latest, but it never replaces or deletes
an older export. No saved results are modified by the supplied browser cells.
